# R1-05 LangSmith monitoring sample (synthetic Spreekorb set)
Makes the AI triage step visible: every draft is traced in LangSmith with input, output, time, tokens and cost, and scored by simple code checks (no AI grading).

**Inputs (read only, hashed before and after):** `data/processed/spreekorb_emails_60.csv`, `langsmith/heldout_labels_v2.csv` (Ayhan's human labels), `n8n/triage_prompt.md` (version read from the file, now v5), `n8n/workflow.json`, `knowledgebase/` PDFs 01-03 and `kb_compiled.md`.

**Creates:** `langsmith/human_review.csv` (the 10 timing_flag drafts), `langsmith/trace_summary.csv` (2 rows: all 20 held-out, the 10 timing_flag; feeds M7 in R1-03 and R1-06), `langsmith/monitoring_note.md`. In LangSmith: datasets `spreekorb-dev-v4` (14 tuning emails) and `spreekorb-heldout-20` (20 held-out emails), experiments `dev-<version>-*` and `heldout-<version>-*` (prefix from the prompt version, now `dev-v5`). The dataset name `spreekorb-dev-v4` stays: same 14 emails, only the experiment is new.

**Old LangSmith experiments stay as history.** The Tobi datasets (`spreekorb-dev-tuning-10*`, `spreekorb-heldout-10*`) and `spreekorb-dev-policy-8` are no longer used by this notebook. Nothing is deleted. `data/processed/synthetic_policy_cases.csv` is left on disk, untouched.

**How to run (Ayhan):**
1. Run all cells with `PROMPT_FROZEN = False`. This makes the dev run (14 emails) and stops before the held-out set.
2. Read the dev table. Freeze the prompt, or change `n8n/triage_prompt.md` (bump the version, update the n8n node) and run again.
3. Make sure every row of `langsmith/heldout_labels_v2.csv` has `queue_human` and `priority_human`. If one is blank the held-out run prints PENDING with the blank ids and stops.
4. Set `PROMPT_FROZEN = True` and `RUN_DEV = False`, run all again. The held-out run happens **once** per prompt version (guard).
5. Fill `score_1_5` in `langsmith/human_review.csv` (note column: no names of people), set `UPLOAD_HUMAN_SCORES = True`, run the upload cell.
6. Add screenshots to `langsmith/screenshots/` and the share link, run the last cells again.

Data: the 60 Spreekorb emails are **synthetic** (invented, generated). No real customer data, no agent names or IDs.

Ground truth: held-out = `queue_human` and `priority_human` (Ayhan), joined on `email_id`. `queue_intended` and `priority_intended` from the emails file are secondary columns. On the dev set the truth is the intended labels.

In [2]:
# 0 Settings and 1 Check: right folder, input files exist, keys present (values are never printed)
from pathlib import Path
import subprocess, shutil, datetime, hashlib, re, json, time
import pandas as pd
from dotenv import dotenv_values, load_dotenv

_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True).stdout.strip()
REPO = Path(_top) if _top else Path.cwd().parent
# the folder is called knowledgebase/ in this repo (triage_prompt.md writes knowledge_base/)
KB = next((REPO / n for n in ['knowledgebase', 'knowledge_base'] if (REPO / n).exists()), REPO / 'knowledgebase')
PROC = REPO / 'data' / 'processed'
LS = REPO / 'langsmith'
NOTEBOOK = REPO / 'notebooks' / 'R1-05_langsmith.ipynb'

# ---- switches for Ayhan
RUN_DEV = False                # dev run on 14 tuning emails (each run makes a new dev experiment)
PROMPT_FROZEN = True         # set True after the STOP cell; then the held-out run happens once
UPLOAD_HUMAN_SCORES = True   # set True after you filled score_1_5 in langsmith/human_review.csv
RUN_UNDO = False              # True restores the backup of the langsmith/ output files

# ---- prices: only used if LangSmith shows no cost. Fill from https://openai.com/api/pricing (gpt-4o-mini)
PRICE_IN_USD_PER_1M = 0.15    # USD per 1M input tokens (gpt-4o-mini)
PRICE_OUT_USD_PER_1M = 0.60   # USD per 1M output tokens (gpt-4o-mini)
PRICE_SOURCE = 'developers.openai.com/api/docs/pricing, standard (non-batch) prices, read 27 Nov 2026'
# ---- USD to EUR for the R1-06 handoff (LangSmith cost is in USD). Fill a dated rate and its source.
USD_TO_EUR = 0.9123           # 1 USD = 0.9123 EUR, given by Ayhan on 27 Nov 2026
USD_TO_EUR_SOURCE = 'tradingview.com (USDEUR), 27 Nov 2026'
LANGSMITH_SHARE_URL = 'https://smith.langchain.com/public/4827265f-9659-41b5-9785-d587b1cd87e8/d'

# ---- names
MODEL = 'gpt-4o-mini'
TEMPERATURE = 0.2
TRACE_NAME = 'spreekorb_triage'
DEV_DATASET = 'spreekorb-dev-v4'
HELDOUT_DATASET = 'spreekorb-heldout-20'
SEED = 42

IN = {
    'pdf01': KB / '01_customer_service_policy.pdf',
    'pdf02': KB / '02_brand_voice_guide.pdf',
    'pdf03': KB / '03_reply_guardrails.pdf',
    'kb_compiled': KB / 'kb_compiled.md',
    'compile_kb': KB / 'compile_kb.py',
    'prompt': REPO / 'n8n' / 'triage_prompt.md',
    'workflow': REPO / 'n8n' / 'workflow.json',
    'emails': PROC / 'spreekorb_emails_60.csv',
    'labels': LS / 'heldout_labels_v2.csv',
    'requirements': REPO / 'requirements.txt',
}
# every file this notebook creates or changes
OUT = {
    'human': LS / 'human_review.csv',
    'trace': LS / 'trace_summary.csv',
    'note': LS / 'monitoring_note.md',
}

results = []
def check(name, ok, detail=''):
    # one PASS/FAIL line, kept in results for the final count
    results.append((name, bool(ok)))
    print(('PASS' if ok else 'FAIL') + ' | ' + name + (' | ' + str(detail) if detail else ''))

check('right folder (repo root has AGENTS.md)', (REPO / 'AGENTS.md').exists(), REPO.name)
for key, p in IN.items():
    check('input exists: ' + str(p.relative_to(REPO)), p.exists())

# .env: presence only. Values are never printed or kept.
ENV_FILE = REPO / '.env'
check('.env file exists', ENV_FILE.exists())
_env = dotenv_values(ENV_FILE) if ENV_FILE.exists() else {}
for k in ['OPENAI_API_KEY', 'LANGSMITH_API_KEY']:
    check(k + ' is set (value not shown)', bool((_env.get(k) or '').strip()))
check('LANGSMITH_TRACING=true', (_env.get('LANGSMITH_TRACING') or '').strip().lower() == 'true')
del _env
load_dotenv(ENV_FILE, override=False)

# requirements.txt: package names only
_req = {re.split(r'[<>=~!\[ ;]', l.strip())[0].lower() for l in IN['requirements'].read_text().splitlines() if l.strip() and not l.strip().startswith('#')}
for pkg in ['langsmith', 'openai', 'pypdf', 'reportlab']:
    check('requirements.txt has ' + pkg, pkg in _req, '' if pkg in _req else 'missing: Ayhan adds it (requirements.txt is not owned by R1-05)')

PROMPT_MD = IN['prompt'].read_text(encoding='utf-8')
PROMPT_VERSION = re.search(r'Version:\s*(v\d+)', PROMPT_MD).group(1)
print('Prompt version:', PROMPT_VERSION)

# the 4 queue names come from the prompt file (lines like:   - "Customer Service": ...)
Q = chr(34)
QUEUES = re.findall(r'^\s+- ' + Q + r'([^' + Q + r']+)' + Q + ':', PROMPT_MD, re.M)
PRIORITIES = ['very_low', 'low', 'medium', 'high', 'critical']
check('4 queue names read from triage_prompt.md', len(QUEUES) == 4, QUEUES)
_cols = pd.read_csv(IN['labels'], nrows=0).columns if IN['labels'].exists() else []
check('labels file has email_id, queue_human, priority_human, note (other columns are ignored)', all(c in _cols for c in ['email_id', 'queue_human', 'priority_human', 'note']))

PASS | right folder (repo root has AGENTS.md) | ecommerce_customersatisfaction_adiadoption
PASS | input exists: knowledgebase/01_customer_service_policy.pdf
PASS | input exists: knowledgebase/02_brand_voice_guide.pdf
PASS | input exists: knowledgebase/03_reply_guardrails.pdf
PASS | input exists: knowledgebase/kb_compiled.md
PASS | input exists: knowledgebase/compile_kb.py
PASS | input exists: n8n/triage_prompt.md
PASS | input exists: n8n/workflow.json
PASS | input exists: data/processed/spreekorb_emails_60.csv
PASS | input exists: langsmith/heldout_labels_v2.csv
PASS | input exists: requirements.txt
PASS | .env file exists
PASS | OPENAI_API_KEY is set (value not shown)
PASS | LANGSMITH_API_KEY is set (value not shown)
PASS | LANGSMITH_TRACING=true
PASS | requirements.txt has langsmith
PASS | requirements.txt has openai
PASS | requirements.txt has pypdf
PASS | requirements.txt has reportlab
Prompt version: v5
PASS | 4 queue names read from triage_prompt.md | ['Customer Service', 'Return

In [3]:
# 2 Backup: copy every output file that already exists to _backup/<timestamp>/, and hash files we do not own
TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BK = REPO / '_backup' / TS
existed = {}
for key, p in OUT.items():
    existed[key] = p.exists()
    if p.exists():
        dest = BK / p.relative_to(REPO)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
print('Already existing (backed up):', [k for k, v in existed.items() if v] or 'none')
print('Backup folder:', BK)

# fingerprint of files this card must not change (.env is never read here)
WATCH = ['knowledgebase', 'knowledge_base', 'n8n', 'data/processed', 'specs', 'research', 'dashboard', 'notebooks', 'langsmith']
WATCH_FILES = ['AGENTS.md', 'README.md', 'board.md', 'requirements.txt', 'CLAUDE.md']
def snapshot():
    files = [REPO / f for f in WATCH_FILES if (REPO / f).exists()]
    for d in WATCH:
        if (REPO / d).exists():
            files += [p for p in (REPO / d).rglob('*') if p.is_file()]
    # our own outputs are not in the fingerprint
    skip = lambda p: p == NOTEBOOK or p in OUT.values() or '.ipynb_checkpoints' in p.parts or p.name == '.DS_Store'
    return {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files if not skip(p)}
BEFORE = snapshot()
print('Files fingerprinted:', len(BEFORE))

# the four read-only inputs named by the card, hashed on their own
INPUT_KEYS = ['emails', 'labels', 'prompt', 'kb_compiled']
BEFORE_IN = {k: hashlib.sha256(IN[k].read_bytes()).hexdigest() for k in INPUT_KEYS if IN[k].exists()}
print('Read-only inputs hashed:', list(BEFORE_IN))

Already existing (backed up): ['human', 'trace', 'note']
Backup folder: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/_backup/20261127_142605
Files fingerprinted: 59
Read-only inputs hashed: ['emails', 'labels', 'prompt', 'kb_compiled']


In [4]:
# 3a Ingest: extract PDFs 01-03 with the same logic as knowledgebase/compile_kb.py (nothing is written)
from pypdf import PdfReader

FOOT = 'Invented for an Ironhack case study. Spreekorb is not a real company. Not legal advice.'
DOCS = ['01_customer_service_policy.pdf', '02_brand_voice_guide.pdf', '03_reply_guardrails.pdf']

def compile_kb(folder):
    parts = []
    for name in DOCS:
        text = '\n'.join(page.extract_text() for page in PdfReader(folder / name).pages)
        lines = [l.strip() for l in text.split('\n')]
        # drop empty lines, the footer and page numbers
        lines = [l for l in lines if l and FOOT not in l and not re.fullmatch(r'Page \d+', l)]
        # a lone bullet glyph becomes "- " in front of the next line
        clean, bullet = [], False
        for l in lines:
            if len(l) == 1 and not l.isalnum():
                bullet = True
                continue
            clean.append(('- ' if bullet else '') + l)
            bullet = False
        parts.append('=== ' + name.replace('.pdf', '') + ' ===\n' + '\n'.join(clean))
    return '\n\n'.join(parts) + '\n'

KB_TEXT = compile_kb(KB)
INGEST_OK = KB_TEXT == IN['kb_compiled'].read_text(encoding='utf-8')
check('ingest: PDFs 01-03 give exactly kb_compiled.md', INGEST_OK, str(len(KB_TEXT.split())) + ' words, sha256 ' + hashlib.sha256(KB_TEXT.encode()).hexdigest()[:12])

PASS | ingest: PDFs 01-03 give exactly kb_compiled.md | 1468 words, sha256 17255209214c


In [5]:
# 3b System prompt: SYSTEM section of triage_prompt.md with {knowledge_base} filled, compared with n8n
SYSTEM_TEMPLATE = PROMPT_MD.split('## SYSTEM\n', 1)[1].split('\n## USER', 1)[0].strip()
USER_TEMPLATE = PROMPT_MD.split('## USER\n', 1)[1].strip()
SYSTEM_PROMPT = SYSTEM_TEMPLATE.replace('{knowledge_base}', KB_TEXT.strip())

WF = json.loads(IN['workflow'].read_text(encoding='utf-8'))
node = lambda name: next(n for n in WF['nodes'] if n['name'] == name)
N8N_SYSTEM = node('Triage + draft')['parameters']['messages']['messageValues'][0]['message']
N8N_USER = node('Triage + draft')['parameters']['text']
N8N_LLM = node('OpenAI gpt-4o-mini')['parameters']
SCHEMA = json.loads(node('Output format')['parameters']['inputSchema'])

PROMPT_OK = SYSTEM_PROMPT == N8N_SYSTEM
check('system prompt equals the n8n "Triage + draft" system message', PROMPT_OK, str(len(SYSTEM_PROMPT)) + ' characters')
# same user message as n8n: "=Subject: {{ $json.subject }}" becomes "Subject: {subject}"
n8n_user = N8N_USER.lstrip('=').replace('{{ $json.subject }}', '{subject}').replace('{{ $json.body }}', '{body}')
check('user message template equals n8n', USER_TEMPLATE == n8n_user, USER_TEMPLATE.replace('\n', ' / '))
check('n8n uses the same model and temperature', N8N_LLM['model']['value'] == MODEL and N8N_LLM['options']['temperature'] == TEMPERATURE)

PASS | system prompt equals the n8n "Triage + draft" system message | 12300 characters
PASS | user message template equals n8n | Subject: {subject} / Email: {body}
PASS | n8n uses the same model and temperature


In [6]:
# 3c Target function: wrapped OpenAI client, gpt-4o-mini, temperature 0.2, JSON output with the n8n schema
from openai import OpenAI
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai

client = Client()               # reads LANGSMITH_API_KEY from the environment
oai = wrap_openai(OpenAI())     # every OpenAI call is traced (tokens, cost, time)

# v5: 'language' (de/en) is the first field, same as the n8n "Output format" node
FIELDS = ['language', 'queue', 'priority', 'reason', 'draft_reply', 'policy_sections']
check('schema from workflow.json has the 6 fields (language first), all required', list(SCHEMA['properties']) == FIELDS and SCHEMA['required'] == FIELDS and SCHEMA['properties']['language'].get('enum') == ['de', 'en'])
# strict JSON mode needs additionalProperties false; fields, types and enums stay the n8n ones
RESPONSE_FORMAT = {'type': 'json_schema', 'json_schema': {'name': 'spreekorb_triage', 'strict': True, 'schema': {**SCHEMA, 'additionalProperties': False}}}

@traceable(name=TRACE_NAME, metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE})
def triage(inputs: dict) -> dict:
    # same cleaning as the n8n "Clean email" node: trim subject and body
    user = USER_TEMPLATE.replace('{subject}', (inputs.get('subject') or '').strip()).replace('{body}', (inputs.get('body') or '').strip())
    resp = oai.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        response_format=RESPONSE_FORMAT,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user}],
    )
    return json.loads(resp.choices[0].message.content)

print('Target ready:', TRACE_NAME, '|', MODEL, '| temperature', TEMPERATURE, '| prompt', PROMPT_VERSION)

PASS | schema from workflow.json has the 6 fields (language first), all required
Target ready: spreekorb_triage | gpt-4o-mini | temperature 0.2 | prompt v5


In [8]:
# 3d Code evaluators only (no AI grading). Each returns a score 1 (ok) or 0 (not ok).
# banned phrases come from the guardrails table in the knowledge base (EN + DE), plus the em dash
_table = KB_TEXT.split('=== 03_reply_guardrails ===')[1].split('English\nGerman\n', 1)[1].split('Style bans:', 1)[0]
BANNED = []
for row in _table.strip().split('\n'):
    row = re.sub(r'\(.*?\)', '', row)              # "(without an alternative)" is a condition, not a phrase
    for piece in row.split(' / '):
        words = [w.strip() for w in piece.split(', ')]
        # "Seamless, delve, leverage" is a word list; "Ich hoffe, diese E-Mail ..." is one phrase
        BANNED += words if all(' ' not in w for w in words) else [piece.strip()]
BANNED = [b for b in BANNED if b] + ['—']   # — is the em dash
print(len(BANNED), 'banned phrases:', BANNED)
check('banned phrase list parsed (EN + DE + em dash)', len(BANNED) >= 25 and 'Rest assured' in BANNED and 'Wie bereits erwähnt' in BANNED, str(len(BANNED)) + ' phrases')

def banned_hits(text):
    t = ' '.join(text.split()).lower()
    return [b for b in BANNED if re.search(r'(?<!\w)' + re.escape(b.lower()) + r'(?!\w)', t) or (b == '—' and b in t)]

# a policy fact = a number of days, working days or hours, or a EUR amount
FACT_RE = re.compile(r'\d+\s*(?:(?:to|bis|-)\s*\d+\s*)?(?:working days|days|day|werktage\w*|tage\w*|tagen|hours|stunden)\b|\beur\b|€', re.I)
SECTION_RE = re.compile(r'Policy ([1-9]|10)')
# stopwords only (v5): sign-off words are left out, signoff_ok checks the sign-off on its own
DE_WORDS = {'und', 'der', 'die', 'das', 'du', 'dein', 'deine', 'dich', 'dir', 'ist', 'nicht', 'wir', 'uns', 'mit', 'für', 'zu', 'ich', 'auf', 'bei', 'eine', 'ein', 'sobald', 'kannst', 'wird', 'werden', 'oder', 'auch', 'dass', 'den', 'dem', 'es'}
EN_WORDS = {'the', 'and', 'you', 'your', 'is', 'we', 'us', 'to', 'for', 'with', 'of', 'this', 'can', 'will', 'our', 'please', 'it', 'be', 'are', 'that', 'in', 'on', 'have'}
# sign-off per language (Brand Voice Guide v1.1)
SIGNOFF = {'de': 'Viele Grüße, dein Spreekorb Kundenservice', 'en': 'Best wishes, Spreekorb Customer Service'}
# topics whose drafts need no policy section (citations_used counts them as pass)
NO_CITATION_TOPICS = ['How-to / product question', 'Thanks / feedback']
# claims about dispatch or check results the AI cannot know (v5 rule: never state a fact you were not given)
UNKNOWN_STATUS = ['noch nicht versendet', 'bereits versendet', 'noch nicht verschickt', 'bereits verschickt', 'ich habe geprüft', 'ich habe überprüft',
                  'not yet dispatched', 'has been dispatched', 'has already been dispatched', 'has not been dispatched', 'I have checked', 'I checked']
ESCALATE_TOPICS = ['Data protection deletion request', 'Legal threat']

def guess_language(text):
    # simple word count: more German function words means German
    words = re.findall(r'[a-zäöüß]+', text.lower())
    de, en = sum(w in DE_WORDS for w in words), sum(w in EN_WORDS for w in words)
    return 'de' if de > en else 'en'

def _draft(o):
    return (o or {}).get('draft_reply') or ''

def queue_match(outputs, reference_outputs):
    return {'key': 'queue_match', 'score': int((outputs or {}).get('queue') == reference_outputs['queue'])}

def priority_match(outputs, reference_outputs):
    return {'key': 'priority_match', 'score': int((outputs or {}).get('priority') == reference_outputs['priority'])}

def priority_within_one(outputs, reference_outputs):
    # one step on the ordered scale very_low < low < medium < high < critical
    p, r = (outputs or {}).get('priority'), reference_outputs['priority']
    ok = p in PRIORITIES and r in PRIORITIES and abs(PRIORITIES.index(p) - PRIORITIES.index(r)) <= 1
    return {'key': 'priority_within_one', 'score': int(ok)}

def no_banned_phrase(outputs):
    hits = banned_hits(_draft(outputs))
    return {'key': 'no_banned_phrase', 'score': int(not hits), 'comment': ', '.join(hits) or 'none'}

def cited_sections_valid(outputs):
    secs = (outputs or {}).get('policy_sections') or []
    bad = [s for s in secs if not SECTION_RE.fullmatch(str(s).strip())]
    return {'key': 'cited_sections_valid', 'score': int(not bad), 'comment': ', '.join(map(str, bad)) or 'ok'}

def policy_fact_cited(outputs):
    facts = FACT_RE.findall(_draft(outputs))
    ok = not facts or bool((outputs or {}).get('policy_sections'))
    return {'key': 'policy_fact_cited', 'score': int(ok), 'comment': 'facts: ' + str(len(facts))}

def language_match(outputs, reference_outputs):
    # the draft's own language (stopword guess on draft_reply) must equal the email's language
    lang = guess_language(_draft(outputs))
    return {'key': 'language_match', 'score': int(lang == reference_outputs['language']), 'comment': 'draft looks ' + lang + ', model said ' + str((outputs or {}).get('language'))}

def signoff_ok(outputs, reference_outputs):
    # the draft ends with the sign-off of the email's language (spaces and line breaks ignored)
    want = SIGNOFF.get(reference_outputs['language'], '')
    end = ' '.join(_draft(outputs).split()).rstrip(' .!')
    return {'key': 'signoff_ok', 'score': int(bool(want) and end.endswith(want)), 'comment': 'want: ' + want}

def no_unknown_status(outputs):
    # 0 if the draft claims a dispatch status or a check result
    t = ' '.join(_draft(outputs).split()).lower()
    hits = [p for p in UNKNOWN_STATUS if re.search(r'(?<!\w)' + re.escape(p.lower()) + r'(?!\w)', t)]
    return {'key': 'no_unknown_status', 'score': int(not hits), 'comment': ', '.join(hits) or 'none'}

def citations_unique(outputs):
    # no section listed twice in policy_sections (e.g. 'Policy 4, Policy 4')
    secs = [str(s).strip() for s in ((outputs or {}).get('policy_sections') or [])]
    dup = sorted({s for s in secs if secs.count(s) > 1})
    return {'key': 'citations_unique', 'score': int(not dup), 'comment': ', '.join(dup) or 'ok'}

def is_escalation(outputs):
    return ((outputs or {}).get('reason') or '').strip().startswith('Escalate to')

def escalation_flagged(outputs, reference_outputs):
    # scored for every topic: the 2 escalation topics must start with 'Escalate to', all other topics must not
    should = reference_outputs.get('topic') in ESCALATE_TOPICS
    did = is_escalation(outputs)
    return {'key': 'escalation_flagged', 'score': int(did == should), 'comment': ('should escalate' if should else 'should not escalate') + ', did ' + ('escalate' if did else 'not escalate')}

def word_count(outputs):
    # v5: 40 to 120 words; holding replies for escalations (reason starts with 'Escalate to'): 40 to 80
    n = len(_draft(outputs).split())
    lo, hi = (40, 80) if is_escalation(outputs) else (40, 120)
    return {'key': 'word_count', 'score': int(lo <= n <= hi), 'comment': str(n) + ' words, allowed ' + str(lo) + '-' + str(hi)}

def citations_used(outputs, reference_outputs):
    # 1 if the draft cites at least one policy section; how-to and thanks topics need none (count as pass)
    if reference_outputs.get('topic') in NO_CITATION_TOPICS:
        return {'key': 'citations_used', 'score': 1, 'comment': 'skipped: ' + reference_outputs.get('topic')}
    return {'key': 'citations_used', 'score': int(bool((outputs or {}).get('policy_sections'))), 'comment': 'escalation' if is_escalation(outputs) else 'normal'}

def queue_intended_match(outputs, reference_outputs):
    # secondary (held-out): compare with the queue the generator intended
    return {'key': 'queue_intended_match', 'score': int((outputs or {}).get('queue') == reference_outputs.get('queue_intended'))}

def priority_intended_match(outputs, reference_outputs):
    # secondary (held-out): compare with the priority the generator intended
    return {'key': 'priority_intended_match', 'score': int((outputs or {}).get('priority') == reference_outputs.get('priority_intended'))}

# S35 fix: if an evaluator raises, LangSmith stores feedback with NO score (a blank cell) and only logs the error.
# safe() turns any error into score 0 with the error text as comment, so a broken check shows as FAIL, never as blank.
import functools
def safe(f):
    @functools.wraps(f)   # keeps the argument names LangSmith reads (outputs, reference_outputs)
    def wrapper(*args, **kwargs):
        try:
            return f(*args, **kwargs)
        except Exception as e:
            return {'key': f.__name__, 'score': 0, 'comment': 'evaluator error: ' + repr(e)[:300]}
    return wrapper

# the 13 evaluators (10 of the card + 3 added in v5: signoff_ok, no_unknown_status, citations_unique)
EVALUATORS = [safe(f) for f in [queue_match, priority_match, priority_within_one, no_banned_phrase, cited_sections_valid, policy_fact_cited, citations_used, escalation_flagged, language_match, word_count, signoff_ok, no_unknown_status, citations_unique]]
EVAL_KEYS = [f.__name__ for f in EVALUATORS]
# held-out: the same 13 + the 2 secondary intended-label columns
HELD_EVALUATORS = EVALUATORS + [safe(queue_intended_match), safe(priority_intended_match)]
HELD_KEYS = [f.__name__ for f in HELD_EVALUATORS]

# quick self-test on made-up text (no API call)
_good = {'queue': 'Billing and Payments', 'priority': 'high', 'reason': 'Refund overdue.', 'policy_sections': ['Policy 5'], 'draft_reply': 'Hi [first name], your refund goes back within 14 days. ' + 'word ' * 50}
_bad = {'queue': 'Billing and Payments', 'priority': 'low', 'reason': 'Refund overdue.', 'policy_sections': [], 'draft_reply': 'Rest assured — refund in 14 days.'}
_ref = {'queue': 'Billing and Payments', 'priority': 'high', 'language': 'en', 'topic': 'Refund overdue'}
check('evaluator self-test: good example passes', no_banned_phrase(_good)['score'] and policy_fact_cited(_good)['score'] and priority_within_one(_good, _ref)['score'] and word_count(_good)['score'] and escalation_flagged(_good, _ref)['score'])
check('evaluator self-test: bad example fails', not no_banned_phrase(_bad)['score'] and not policy_fact_cited(_bad)['score'] and not priority_within_one(_bad, _ref)['score'] and not word_count(_bad)['score'])
check('evaluator self-test: Policy 11 is not a valid section', cited_sections_valid({'policy_sections': ['Policy 11']})['score'] == 0 and cited_sections_valid({'policy_sections': ['Policy 10']})['score'] == 1)
_esc = {'reason': 'Escalate to Data Protection Officer: deletion request.', 'draft_reply': 'word ' * 50}
check('evaluator self-test: 90 words ok for a normal reply, not for escalation; 30 words never ok', word_count({'reason': 'Normal.', 'draft_reply': 'word ' * 90})['score'] == 1 and word_count({'reason': _esc['reason'], 'draft_reply': 'word ' * 90})['score'] == 0 and word_count({'reason': 'Normal.', 'draft_reply': 'word ' * 30})['score'] == 0)
_t = {'topic': 'Legal threat'}
check('evaluator self-test: escalation_flagged both ways', escalation_flagged(_esc, _t)['score'] == 1 and escalation_flagged(_good, _t)['score'] == 0 and escalation_flagged(_esc, _ref)['score'] == 0)
_de = {'draft_reply': 'Hallo [Vorname], wir prüfen das und melden uns bei dir.\n\nViele Grüße,\ndein Spreekorb Kundenservice'}
_en = {'draft_reply': 'Hi [first name], we will check this and get back to you.\n\nBest wishes,\nSpreekorb Customer Service'}
check('evaluator self-test: language_match and signoff_ok (de and en, line breaks ok)', language_match(_de, {'language': 'de'})['score'] == 1 and language_match(_en, {'language': 'en'})['score'] == 1 and language_match(_en, {'language': 'de'})['score'] == 0 and signoff_ok(_de, {'language': 'de'})['score'] == 1 and signoff_ok(_en, {'language': 'en'})['score'] == 1 and signoff_ok(_en, {'language': 'de'})['score'] == 0)
check('evaluator self-test: no_unknown_status catches dispatch / check claims', no_unknown_status({'draft_reply': 'Deine Bestellung wurde noch nicht versendet.'})['score'] == 0 and no_unknown_status({'draft_reply': 'I have checked your order.'})['score'] == 0 and no_unknown_status(_en)['score'] == 1)
check('evaluator self-test: citations_unique and citations_used skip', citations_unique({'policy_sections': ['Policy 4', 'Policy 4']})['score'] == 0 and citations_unique({'policy_sections': ['Policy 4', 'Policy 6']})['score'] == 1 and citations_used({'policy_sections': []}, {'topic': 'Thanks / feedback'})['score'] == 1 and citations_used({'policy_sections': []}, {'topic': 'Refund overdue'})['score'] == 0)
check('evaluator self-test: safe() turns an evaluator error into score 0, not blank', EVALUATORS[EVAL_KEYS.index('language_match')](None, {})['score'] == 0)

29 banned phrases: ['I hope this email finds you well', 'Ich hoffe, diese E-Mail erreicht dich gut', 'Rest assured', 'Seien Sie versichert', 'We apologize for any inconvenience caused', 'Wir entschuldigen uns für etwaige Unannehmlichkeiten', 'Thank you for reaching out to us', 'Vielen Dank, dass Sie sich an uns gewandt haben', 'Seamless', 'delve', 'leverage', 'elevate', 'nahtlos', 'maßgeschneidert', 'ganzheitlich', 'As I already said', 'As mentioned before', 'Wie bereits erwähnt', 'Unfortunately that is not possible', 'Leider nicht möglich', 'Your fault', 'you should have', 'Ihr Fehler', 'Sie hätten', 'We guarantee', '100%', 'Wir garantieren', '100 %', '—']
PASS | banned phrase list parsed (EN + DE + em dash) | 29 phrases
PASS | evaluator self-test: good example passes
PASS | evaluator self-test: bad example fails
PASS | evaluator self-test: Policy 11 is not a valid section
PASS | evaluator self-test: 90 words ok for a normal reply, not for escalation; 30 words never ok
PASS | evaluato

In [9]:
# 3e Datasets: 14 dev emails (tuning split, one per topic, seed 42) and the 20 held-out emails
em = pd.read_csv(IN['emails'], dtype=str).fillna('')   # pandas 3: fill blanks before using text
em['timing_flag'] = em['timing_flag'].str.strip().str.lower().eq('true')
for c in ['subject', 'body', 'topic', 'language']:
    em[c] = em[c].str.strip()
tun = em[em['split'] == 'tuning'].reset_index(drop=True)
held_df = em[em['split'] == 'heldout'].sort_values('email_id').reset_index(drop=True)

# dev: one email per topic. Seeded retry: seeds 42, 43, ... until the mix is 7 de / 7 en (or keep the closest)
best = None
for s in range(SEED, SEED + 200):
    pick = tun.groupby('topic').sample(1, random_state=s)
    gap = abs(int((pick['language'] == 'de').sum()) - 7)
    if best is None or gap < best[0]:
        best = (gap, s, pick)
    if gap == 0:
        break
DEV_SEED = best[1]
dev_df = best[2].sort_values('email_id').reset_index(drop=True)
print('Dev sample: seed', DEV_SEED, '| language mix', dev_df['language'].value_counts().to_dict())

# held-out truth: queue_human and priority_human from heldout_labels_v2.csv, joined on email_id (other columns of that file are ignored)
lab = pd.read_csv(IN['labels'], dtype=str).fillna('')
lab = lab[[c for c in ['email_id', 'queue_human', 'priority_human', 'note'] if c in lab.columns]]
held_df = held_df.merge(lab, on='email_id', how='left')
for c in ['queue_human', 'priority_human']:
    held_df[c] = held_df[c].fillna('').str.strip()
BLANK_IDS = held_df.loc[~held_df['queue_human'].isin(QUEUES) | ~held_df['priority_human'].isin(PRIORITIES), 'email_id'].tolist()
LABELS_OK = len(held_df) == 20 and not BLANK_IDS

check('dev: 14 emails, 14 topics, all from the tuning split', len(dev_df) == 14 and dev_df['topic'].nunique() == 14 and set(dev_df['split']) == {'tuning'}, dev_df['language'].value_counts().to_dict())
check('dev: language mix within one of 7 de / 7 en', abs(int((dev_df['language'] == 'de').sum()) - 7) <= 1, 'seed ' + str(DEV_SEED))
check('held-out: the 20 heldout emails, 10 with timing_flag', len(held_df) == 20 and int(held_df['timing_flag'].sum()) == 10, str(len(held_df)) + ' emails')
check('no overlap between dev and held-out', not set(dev_df['email_id']) & set(held_df['email_id']))

def make_examples(df, qcol, pcol, source):
    # inputs = what the model sees; outputs = labels for the evaluators (queue and priority = the truth for this set)
    ex = []
    for _, r in df.iterrows():
        ex.append({
            'inputs': {'email_id': r['email_id'], 'subject': r['subject'], 'body': r['body']},
            'outputs': {'queue': r[qcol], 'priority': r[pcol], 'queue_intended': r['queue_intended'], 'priority_intended': r['priority_intended'], 'language': r['language'], 'topic': r['topic']},
            'metadata': {'email_id': r['email_id'], 'split': r['split'], 'source': source},
        })
    return ex

def get_or_create_dataset(name, examples, description):
    # reuse the dataset if it exists, but only if it holds exactly the same emails
    want = sorted(e['inputs']['email_id'] for e in examples)
    if client.has_dataset(dataset_name=name):
        ds = client.read_dataset(dataset_name=name)
        ids = sorted((e.metadata or {}).get('email_id') for e in client.list_examples(dataset_id=ds.id))
        check(name + ': existing dataset has the same ' + str(len(want)) + ' emails', ids == want, ids)
        return ds
    ds = client.create_dataset(name, description=description)
    client.create_examples(dataset_id=ds.id, examples=examples)
    check(name + ': dataset created with ' + str(len(want)) + ' emails', len(list(client.list_examples(dataset_id=ds.id))) == len(want))
    return ds

dev_ds = get_or_create_dataset(DEV_DATASET, make_examples(dev_df, 'queue_intended', 'priority_intended', 'synthetic Spreekorb (R1-02b), intended labels'), 'R1-05 dev: 14 synthetic Spreekorb tuning emails, one per topic, seed 42. Truth = intended labels.')
# the held-out dataset is created in 3g, only after all 20 human labels are valid

def experiments(dataset, prefix):
    # experiments of this dataset whose name starts with the prefix, newest first (none if the dataset does not exist yet)
    if not client.has_dataset(dataset_name=dataset):
        return []
    ps = [p for p in client.list_projects(reference_dataset_name=dataset) if p.name.startswith(prefix + '-')]
    return sorted(ps, key=lambda p: p.start_time, reverse=True)

def load_experiment(name, n_expected=14, n_keys=len(EVAL_KEYS), tries=12):
    # read the experiment back from LangSmith: root runs (time, input, output), LLM runs (tokens, cost), feedback (scores)
    for _ in range(tries):
        roots = [r for r in client.list_runs(project_name=name, is_root=True) if r.name == TRACE_NAME]
        llms = list(client.list_runs(project_name=name, run_type='llm'))
        fb = list(client.list_feedback(run_ids=[r.id for r in roots])) if roots else []
        # ready = every run has n_keys feedback rows WITH a score (a total count can hide one run with blanks, as S35 in dev-v4)
        scored = {}
        for f in fb:
            if f.score is not None:
                scored.setdefault(str(f.run_id), set()).add(f.key)
        ready = len(roots) >= n_expected and all(r.end_time for r in roots) and len(llms) >= n_expected and all(len(scored.get(str(r.id), ())) >= n_keys for r in roots)
        if ready:
            break
        time.sleep(10)   # LangSmith needs a few seconds to process tokens, cost and scores
    by_trace = {}
    for r in llms:
        d = by_trace.setdefault(r.trace_id, {'tokens_in': 0, 'tokens_out': 0, 'cost_usd': 0.0, 'cost_missing': False})
        d['tokens_in'] += r.prompt_tokens or 0
        d['tokens_out'] += r.completion_tokens or 0
        if r.total_cost is None:
            d['cost_missing'] = True
        else:
            d['cost_usd'] += float(r.total_cost)
    rows = []
    for r in roots:
        inp = (r.inputs or {}).get('inputs', r.inputs or {})
        out = r.outputs or {}
        u = by_trace.get(r.trace_id, {'tokens_in': None, 'tokens_out': None, 'cost_usd': None, 'cost_missing': True})
        rows.append({
            'email_id': inp.get('email_id'), 'run_id': str(r.id), 'error': r.error,
            'language_ai': out.get('language'), 'queue_ai': out.get('queue'), 'priority_ai': out.get('priority'), 'reason': out.get('reason'),
            'draft_reply': out.get('draft_reply'), 'policy_sections': ', '.join(out.get('policy_sections') or []),
            'latency_s': (r.end_time - r.start_time).total_seconds() if r.end_time else None,
            'tokens_in': u['tokens_in'], 'tokens_out': u['tokens_out'],
            'cost_usd': None if u['cost_missing'] else u['cost_usd'],
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    scores = pd.DataFrame([{'run_id': str(f.run_id), 'key': f.key, 'score': f.score, 'comment': f.comment} for f in fb])
    # show every blank score with the reason LangSmith stored (an evaluator error is kept in the comment)
    _email = dict(zip(df['run_id'], df['email_id']))
    _blank = scores[scores['score'].isna()] if not scores.empty else scores
    for _, b in _blank.iterrows():
        print('BLANK score:', _email.get(b['run_id']), b['key'], '|', str(b['comment'])[:200])
    if not scores.empty:
        df = df.merge(scores.pivot_table(index='run_id', columns='key', values='score', aggfunc='last').reset_index(), on='run_id', how='left')
    return df.sort_values('email_id').reset_index(drop=True)

Dev sample: seed 44 | language mix {'de': 7, 'en': 7}
PASS | dev: 14 emails, 14 topics, all from the tuning split | {'de': 7, 'en': 7}
PASS | dev: language mix within one of 7 de / 7 en | seed 44
PASS | held-out: the 20 heldout emails, 10 with timing_flag | 20 emails
PASS | no overlap between dev and held-out
PASS | spreekorb-dev-v4: existing dataset has the same 14 emails | ['S03', 'S10', 'S14', 'S17', 'S22', 'S26', 'S31', 'S35', 'S39', 'S43', 'S48', 'S54', 'S56', 'S59']


**Note: old LangSmith experiments stay as history.** The Tobi datasets (`spreekorb-dev-tuning-10*`, `spreekorb-heldout-10*`) and `spreekorb-dev-policy-8`, with their experiments, are not deleted and not used any more. This notebook only works with `spreekorb-dev-v4` and `spreekorb-heldout-20`. The file `data/processed/synthetic_policy_cases.csv` is also left as it is.

**S35 blank scores in dev-v4 (likely cause, not confirmed from the repo: the saved notebook table shows S35 scored):** when an evaluator raises an error, LangSmith (SDK 0.14) stores that feedback without a score and only writes the error to the log, so the LangSmith cell is blank. The read-back check only counted feedback rows in total, and an error row counts too, so it did not notice. Fix (v5): every evaluator is wrapped in `safe()` (an error becomes score 0 with the error text as comment), the read-back waits until every run has all scores, and any blank is printed with its reason.

In [10]:
# 3f Dev run: 14 tuning emails, experiment prefix dev-<version>. Safe to repeat while tuning.
DEV_PREFIX = 'dev-' + PROMPT_VERSION
if RUN_DEV and PROMPT_FROZEN:
    print('PROMPT_FROZEN = True: no new dev run (the prompt is frozen). Set RUN_DEV = False to silence this.')
if RUN_DEV and not PROMPT_FROZEN and INGEST_OK and PROMPT_OK:
    dev_res = client.evaluate(
        triage, data=DEV_DATASET, evaluators=EVALUATORS, experiment_prefix=DEV_PREFIX,
        description='R1-05 dev run, prompt ' + PROMPT_VERSION, max_concurrency=1,   # one at a time = clean latency
        metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE, 'set': 'dev'},
    )
    DEV_EXP = dev_res.experiment_name
elif RUN_DEV and not PROMPT_FROZEN:
    DEV_EXP = None
    print('Dev run skipped: ingest or prompt check failed (fix that first).')
else:
    _old = experiments(DEV_DATASET, DEV_PREFIX)
    DEV_EXP = _old[0].name if _old else None
    print('RUN_DEV = False, reusing the newest dev experiment:', DEV_EXP)

dev_out = load_experiment(DEV_EXP, n_expected=14, n_keys=len(EVAL_KEYS)) if DEV_EXP else pd.DataFrame()
dev_table = pd.DataFrame()
if not dev_out.empty:
    ref = dev_df[['email_id', 'topic', 'language', 'queue_intended', 'priority_intended']]
    dev_table = ref.merge(dev_out, on='email_id', how='left')
    dev_table['words'] = dev_table['draft_reply'].fillna('').astype(str).str.split().str.len()
    show = ['email_id', 'topic', 'language', 'language_ai', 'queue_intended', 'queue_ai', 'priority_intended', 'priority_ai', 'policy_sections', 'words', 'latency_s'] + [k for k in EVAL_KEYS if k in dev_table]
    pd.set_option('display.max_colwidth', 40)
    print('Experiment:', DEV_EXP)
    display(dev_table[show])
    print('Experiment prefix:', DEV_PREFIX, '(from the prompt version)')
    print('Share of 14 passing per evaluator:')
    print(dev_table[[k for k in EVAL_KEYS if k in dev_table]].mean().round(2).to_string())

RUN_DEV = False, reusing the newest dev experiment: dev-v5-617203f1


/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_44719/1845986704.py:73: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  roots = [r for r in client.list_runs(project_name=name, is_root=True) if r.name == TRACE_NAME]
/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_44719/1845986704.py:74: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  llms = list(client.list_runs(project_name=name, run_type='llm'))


Experiment: dev-v5-617203f1


,email_id,topic,language,language_ai,queue_intended,queue_ai,priority_intended,priority_ai,policy_sections,words,...,no_banned_phrase,cited_sections_valid,policy_fact_cited,citations_used,escalation_flagged,language_match,word_count,signoff_ok,no_unknown_status,citations_unique
0,S03,Order status,de,de,Customer Service,Customer Service,medium,medium,"Policy 2, Policy 8",40,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,S10,"Late delivery, no tracking update",en,en,Customer Service,Customer Service,medium,high,"Policy 2, Policy 8",65,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,S14,Cancellation before dispatch,en,en,Customer Service,Customer Service,medium,high,Policy 3,49,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,S17,Return request,de,de,Returns and Exchanges,Returns and Exchanges,medium,medium,"Policy 4, Policy 4, Policy 5",67,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4,S22,"Exchange, wrong size",en,en,Returns and Exchanges,Returns and Exchanges,medium,medium,"Policy 4, Policy 5",82,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN
5,S26,Wrong item received,en,en,Returns and Exchanges,Returns and Exchanges,medium,high,"Policy 4, Policy 6",88,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,S31,Refund overdue,de,de,Billing and Payments,Billing and Payments,high,high,"Policy 5, Policy 8",48,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,S35,Invoice resend,de,de,Billing and Payments,Billing and Payments,medium,medium,Policy 7,47,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,S39,Double charge,de,de,Billing and Payments,Billing and Payments,high,high,"Policy 7, Policy 8",44,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,S43,Damaged item,de,de,Product Support,Returns and Exchanges,high,high,Policy 6,52,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Experiment prefix: dev-v5 (from the prompt version)
Share of 14 passing per evaluator:
queue_match             0.92
priority_match          0.71
priority_within_one     1.00
no_banned_phrase        1.00
cited_sections_valid    1.00
policy_fact_cited       1.00
citations_used          1.00
escalation_flagged      1.00
language_match          1.00
word_count              1.00
signoff_ok              1.00
no_unknown_status       1.00
citations_unique        0.92


## STOP
**Freeze the prompt, or change `n8n/triage_prompt.md` (bump version) and re-run dev.**

If you change the prompt: edit `triage_prompt.md` (new version), paste the new system message into the n8n "Triage + draft" node, export `workflow.json`, then run all again. The prompt check fails until both match.
Never tune on the held-out emails. When the prompt is frozen, set `PROMPT_FROZEN = True` and `RUN_DEV = False` in cell 1 and run all again.

In [11]:
# STOP cell: read the dev table above before you go on
print('STOP: Freeze the prompt, or change triage_prompt.md (bump version) and re-run dev.')
print('Prompt version:', PROMPT_VERSION, '| PROMPT_FROZEN =', PROMPT_FROZEN)
if not PROMPT_FROZEN:
    print('The held-out run will be skipped. Set PROMPT_FROZEN = True in cell 1 when the prompt is frozen.')

STOP: Freeze the prompt, or change triage_prompt.md (bump version) and re-run dev.
Prompt version: v5 | PROMPT_FROZEN = True


In [12]:
# 3g Official run: the 20 held-out emails, prefix heldout-<version>. Runs ONCE per prompt version.
HELD_PREFIX = 'heldout-' + PROMPT_VERSION
held_table = pd.DataFrame()
_done = experiments(HELDOUT_DATASET, HELD_PREFIX)
if not LABELS_OK:
    HELD_EXP = None
    print('PENDING: queue_human or priority_human is blank or invalid for', len(BLANK_IDS), 'email(s):', BLANK_IDS)
    print('Queues:', QUEUES, '| priorities:', PRIORITIES)
    print('Fix langsmith/heldout_labels_v2.csv. Held-out run stopped here.')
elif _done:
    HELD_EXP = _done[0].name
    print('Guard: held-out experiment for', PROMPT_VERSION, 'already exists (' + HELD_EXP + '). Not running again, reading it back.')
    print('If that run failed part-way, delete it in LangSmith, then run this cell again.')
elif PROMPT_FROZEN and INGEST_OK and PROMPT_OK:
    held_ds = get_or_create_dataset(HELDOUT_DATASET, make_examples(held_df, 'queue_human', 'priority_human', 'synthetic Spreekorb (R1-02b), human labels'), 'R1-05 held-out: 20 synthetic Spreekorb emails. Truth = queue_human and priority_human (Ayhan). Run once per prompt version.')
    # an older dataset without the human labels must not be used
    _ex = {(e.metadata or {}).get('email_id'): e.outputs or {} for e in client.list_examples(dataset_id=held_ds.id)}
    DS_LABELS_OK = all(_ex.get(r['email_id'], {}).get('queue') == r['queue_human'] and _ex.get(r['email_id'], {}).get('priority') == r['priority_human'] for _, r in held_df.iterrows())
    check('held-out dataset holds the human labels', DS_LABELS_OK, 'if FAIL: delete ' + HELDOUT_DATASET + ' in LangSmith and run again')
    if DS_LABELS_OK:
        held_res = client.evaluate(
            triage, data=HELDOUT_DATASET, evaluators=HELD_EVALUATORS, experiment_prefix=HELD_PREFIX,
            description='R1-05 official held-out run, prompt ' + PROMPT_VERSION + ' (frozen), queue and priority vs human labels', max_concurrency=1,
            metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE, 'set': 'heldout', 'truth': 'queue_human, priority_human'},
        )
        HELD_EXP = held_res.experiment_name
    else:
        HELD_EXP = None
else:
    HELD_EXP = None
    print('Held-out run skipped: PROMPT_FROZEN is False, or the ingest/prompt check failed.')

held_out = load_experiment(HELD_EXP, n_expected=20, n_keys=len(HELD_KEYS)) if HELD_EXP else pd.DataFrame()
if not held_out.empty:
    ref = held_df[['email_id', 'topic', 'language', 'subject', 'timing_flag', 'queue_human', 'priority_human', 'queue_intended', 'priority_intended']]
    held_table = ref.merge(held_out, on='email_id', how='left')
    print('Experiment:', HELD_EXP)
    display(held_table[['email_id', 'topic', 'language', 'queue_human', 'queue_ai', 'priority_human', 'priority_ai', 'queue_intended', 'priority_intended', 'policy_sections', 'latency_s'] + [k for k in HELD_KEYS if k in held_table]])

Guard: held-out experiment for v5 already exists (heldout-v5-489260bd). Not running again, reading it back.
If that run failed part-way, delete it in LangSmith, then run this cell again.


/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_44719/1845986704.py:73: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  roots = [r for r in client.list_runs(project_name=name, is_root=True) if r.name == TRACE_NAME]
/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_44719/1845986704.py:74: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  llms = list(client.list_runs(project_name=name, run_type='llm'))


Experiment: heldout-v5-489260bd


,email_id,topic,language,queue_human,queue_ai,priority_human,priority_ai,queue_intended,priority_intended,policy_sections,...,policy_fact_cited,citations_used,escalation_flagged,language_match,word_count,signoff_ok,no_unknown_status,citations_unique,queue_intended_match,priority_intended_match
0,S05,Order status,de,Customer Service,Customer Service,low,medium,Customer Service,medium,"Policy 2, Policy 8",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,S06,"Late delivery, no tracking update",en,Customer Service,Customer Service,high,high,Customer Service,medium,Policy 2,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
2,S07,"Late delivery, no tracking update",de,Customer Service,Customer Service,medium,high,Customer Service,medium,Policy 2,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
3,S11,Cancellation before dispatch,de,Customer Service,Customer Service,high,high,Customer Service,medium,"Policy 3, Policy 2",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4,S15,Return request,de,Returns and Exchanges,Returns and Exchanges,medium,medium,Returns and Exchanges,medium,"Policy 4, Policy 5, Policy 6",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,S16,Return request,en,Product Support,Returns and Exchanges,critical,medium,Returns and Exchanges,medium,"Policy 4, Policy 5, Policy 6",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,S20,"Exchange, wrong size",en,Returns and Exchanges,Returns and Exchanges,medium,medium,Returns and Exchanges,medium,"Policy 4, Policy 5",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,S24,Wrong item received,en,Returns and Exchanges,Returns and Exchanges,high,high,Returns and Exchanges,medium,"Policy 4, Policy 6, Policy 5",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
8,S30,Refund overdue,en,Billing and Payments,Billing and Payments,high,high,Billing and Payments,high,"Policy 5, Policy 8",...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,S33,Invoice resend,de,Billing and Payments,Billing and Payments,medium,medium,Billing and Payments,medium,Policy 7,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [13]:
# 3g-b Backfill: score again ONLY the (email, evaluator) pairs LangSmith did not store (e.g. S22, S34).
# Code checks give the same result every time on the same draft, so this is not a re-run: no new drafts, no API cost.
import inspect
EVAL_FN = {f.__name__: f for f in HELD_EVALUATORS}

def backfill(table, keys, label):
    done = 0
    for i, r in table.iterrows():
        miss = [k for k in keys if k not in table or pd.isna(r[k])]
        if not miss:
            continue
        run = client.read_run(r['run_id'])                       # the stored draft (outputs) of this email
        ex = client.read_example(run.reference_example_id)       # the same labels the evaluators used
        have = {f.key: f.score for f in client.list_feedback(run_ids=[run.id]) if f.score is not None}
        for k in miss:
            if k in have:                                         # LangSmith has it by now: just use it
                table.loc[i, k] = have[k]
                continue
            f = EVAL_FN[k]
            res = f(run.outputs, ex.outputs) if 'reference_outputs' in inspect.signature(f).parameters else f(run.outputs)
            client.create_feedback(run.id, key=k, score=res['score'], comment='backfill (LangSmith had no score): ' + str(res.get('comment', ''))[:300])
            table.loc[i, k] = res['score']
            print('Backfilled', label, r['email_id'], k, '=', res['score'])
            done += 1
    return done

n_dev = backfill(dev_table, EVAL_KEYS, 'dev') if not dev_table.empty else 0
n_held = backfill(held_table, HELD_KEYS, 'held-out') if not held_table.empty else 0
print('Backfilled scores:', n_dev, 'dev,', n_held, 'held-out (0 and 0 = nothing was missing)')

/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_44719/1343566942.py:12: DeprecationWarning: read_run() is deprecated and will be removed after Jan 31, 2027. Use client.runs.retrieve() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-retrieve for the migration guide.
  run = client.read_run(r['run_id'])                       # the stored draft (outputs) of this email


Backfilled scores: 0 dev, 0 held-out (0 and 0 = nothing was missing)


In [14]:
# 3h Human review sheet for Ayhan: langsmith/human_review.csv (the 10 timing_flag drafts, score_1_5 left blank)
LS.mkdir(parents=True, exist_ok=True)
HR_COLS = ['email_id', 'language', 'subject', 'draft_reply', 'score_1_5', 'note']
def has_scores(p):
    # True if Ayhan already typed scores: then we never overwrite the file
    if not p.exists():
        return False
    s = pd.read_csv(p, dtype=str).get('score_1_5', pd.Series(dtype=str))
    return s.fillna('').str.strip().ne('').any()

if held_table.empty:
    print('PENDING: human_review.csv is written after the held-out run.')
elif has_scores(OUT['human']):
    print('human_review.csv already has your scores, not overwritten.')
else:
    hr = held_table[held_table['timing_flag']][['email_id', 'language', 'subject', 'draft_reply']].copy()
    hr['draft_reply'] = hr['draft_reply'].fillna('')
    hr['score_1_5'] = ''
    hr['note'] = ''
    hr[HR_COLS].to_csv(OUT['human'], index=False)
    print('Wrote', OUT['human'].relative_to(REPO), len(hr), 'rows. Fill score_1_5 (1 = unusable, 5 = send as is) and note.')

human_review.csv already has your scores, not overwritten.


In [15]:
# 3h (optional) Upload your filled scores to LangSmith as feedback 'human_score' (runs only if UPLOAD_HUMAN_SCORES = True)
if UPLOAD_HUMAN_SCORES and not held_out.empty and OUT['human'].exists():
    filled = pd.read_csv(OUT['human'], dtype=str).fillna('')
    run_of = dict(zip(held_out['email_id'], held_out['run_id']))
    done = 0
    for _, r in filled.iterrows():
        s = r['score_1_5'].strip()
        if s not in {'1', '2', '3', '4', '5'} or r['email_id'] not in run_of:
            continue
        rid = run_of[r['email_id']]
        # skip if this run already has a human_score (no double upload)
        if any(f.key == 'human_score' for f in client.list_feedback(run_ids=[rid])):
            continue
        client.create_feedback(rid, key='human_score', score=int(s), comment=r['note'][:500])
        done += 1
    print('Uploaded human_score for', done, 'drafts.')
else:
    print('Upload skipped (UPLOAD_HUMAN_SCORES = False, or no held-out run yet).')

Uploaded human_score for 0 drafts.


In [16]:
# 3i Trace summary: langsmith/trace_summary.csv. Row heldout_20 = all 20 held-out, row timing_10 = the 10 timing_flag emails.
# R1-03 M7 needs: drafts traced = n, seconds per draft = avg_latency_s, cost per draft in EUR = total_cost_eur / n.
TS_COLS = ['set', 'n', 'avg_latency_s', 'avg_tokens_in', 'avg_tokens_out', 'total_cost_usd', 'total_cost_eur']
COST_SOURCE = 'n/a'
def summarize(df, name):
    global COST_SOURCE
    n = int(df['error'].isna().sum())   # drafts traced without an error
    if df['cost_usd'].notna().all():
        usd, COST_SOURCE = float(df['cost_usd'].sum()), 'LangSmith run cost (measured, USD)'
    elif PRICE_IN_USD_PER_1M is not None and PRICE_OUT_USD_PER_1M is not None:
        usd = float((df['tokens_in'] * PRICE_IN_USD_PER_1M + df['tokens_out'] * PRICE_OUT_USD_PER_1M).sum() / 1e6)
        COST_SOURCE = 'token usage x OpenAI list prices (' + PRICE_SOURCE + ')'
    else:
        usd, COST_SOURCE = None, 'PENDING: set PRICE_IN/OUT_USD_PER_1M in cell 1'
    eur = round(usd * USD_TO_EUR, 6) if (usd is not None and USD_TO_EUR) else 'PENDING'
    return {'set': name, 'n': n, 'avg_latency_s': round(df['latency_s'].mean(), 2),
            'avg_tokens_in': round(df['tokens_in'].mean(), 1), 'avg_tokens_out': round(df['tokens_out'].mean(), 1),
            'total_cost_usd': round(usd, 6) if usd is not None else 'PENDING', 'total_cost_eur': eur}

trace_summary = pd.DataFrame()
if not held_table.empty:
    trace_summary = pd.DataFrame([summarize(held_table, 'heldout_20'), summarize(held_table[held_table['timing_flag']], 'timing_10')])[TS_COLS]
    trace_summary.to_csv(OUT['trace'], index=False)
    print('Wrote', OUT['trace'].relative_to(REPO))
    print('EUR = USD x', USD_TO_EUR, '(' + USD_TO_EUR_SOURCE + '): the EUR figure is an estimate. Cost source:', COST_SOURCE)
    display(trace_summary)
else:
    print('No held-out experiment yet, trace_summary.csv not written.')

Wrote langsmith/trace_summary.csv
EUR = USD x 0.9123 (tradingview.com (USDEUR), 27 Nov 2026): the EUR figure is an estimate. Cost source: LangSmith run cost (measured, USD)


,set,n,avg_latency_s,avg_tokens_in,avg_tokens_out,total_cost_usd,total_cost_eur
0,heldout_20,20,2.21,3099.2,129.8,0.007591,0.006925
1,timing_10,10,2.22,3107.2,128.9,0.003802,0.003469


In [17]:
# 3j Monitoring note: langsmith/monitoring_note.md (all numbers come from the experiments above)
SHOTS = LS / 'screenshots'
shots = sorted(p for p in SHOTS.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}) if SHOTS.exists() else []
pct = lambda df, k: (str(int(round(df[k].mean() * 100))) + '%') if (not df.empty and k in df) else 'n/a'
ts = {r['set']: r for r in trace_summary.to_dict('records')} if not trace_summary.empty else {}

L = []
add = L.append
add('# LangSmith monitoring note: R1-05')
add('')
add('Prompt ' + PROMPT_VERSION + ' (n8n/triage_prompt.md), model ' + MODEL + ', temperature ' + str(TEMPERATURE) + '. Written by notebooks/R1-05_langsmith.ipynb.')
add('')
add('## What was monitored')
add('- Every AI triage + draft call is traced in LangSmith as ' + TRACE_NAME + ': the email in, the JSON out (language, queue, priority, reason, draft reply, policy sections), time, tokens and cost.')
add('- Dev set: ' + DEV_DATASET + ' (14 tuning emails, one per topic, seed ' + str(DEV_SEED) + '), experiment ' + str(DEV_EXP) + '. Truth = intended labels.')
add('- Official held-out set: ' + HELDOUT_DATASET + ' (20 emails, never used for tuning), experiment ' + str(HELD_EXP or 'PENDING') + '. Run once per prompt version. Truth = human labels (queue_human, priority_human) by Ayhan; intended labels are secondary columns.')
add('- Code checks only, no AI grading. A person scores the 10 timing_flag drafts 1 to 5 (human_review.csv).')
add('- Old LangSmith experiments (Tobi datasets, policy-8) stay as history and are not used.')
add('')
add('## Why the set is synthetic')
add('- The public Tobi-Bueck tickets turned out to be SaaS and IT topics: 0 of 50 fit Spreekorb, by manual review (Ayhan).')
add('- So the test emails are 60 invented Spreekorb emails (spreekorb_emails_60.csv, synthetic, generated): 14 topics, German and English, tuning and held-out split.')
add('')
add('## Prompt history v1 to v5')
add('- v1 and v2: earlier versions. The version notes in n8n/triage_prompt.md do not record what changed, so nothing is claimed here.')
add('- v3: added queue definitions, a priority rubric (aligned with Policy 8) and a length rule.')
add('- v4 (27 Nov 2026): after dev run 2, added the reply language rule: an English email gets an English reply, a German email a German reply with du.')
add('- v5 (27 Nov 2026): after dev-v4, language field first in the output, length 40 to 120 words, no unknown facts (no dispatch status or check results), each policy section cited once. Brand Voice Guide v1.1.')
add('- Rule: the prompt is never tuned on the held-out emails. A change gets a new version and a new experiment.')
add('')
add('## Held-out results per evaluator (share of drafts passing)')
add('')
add('| Evaluator | What it checks | Dev (14) | Held-out (20) |')
add('|---|---|---|---|')
WHAT = {
    'queue_match': 'queue equals the truth (held-out: human label)', 'priority_match': 'priority equals the truth (held-out: human label)',
    'priority_within_one': 'priority at most one step off', 'no_banned_phrase': 'no banned phrase (EN/DE guardrails) and no em dash',
    'cited_sections_valid': 'cited sections are Policy 1 to 10', 'policy_fact_cited': 'a draft with days, hours or EUR cites a policy section',
    'citations_used': 'draft cites at least one policy section (how-to and thanks topics pass)', 'escalation_flagged': 'deletion request and legal threat start with Escalate to; other topics do not',
    'language_match': 'draft in the email language (stopword guess on the draft)', 'word_count': '40 to 120 words (40 to 80 for escalation holding replies)',
    'signoff_ok': 'draft ends with the sign-off of the email language', 'no_unknown_status': 'no claim of dispatch status or check results',
    'citations_unique': 'no policy section listed twice',
    'queue_intended_match': 'secondary: queue equals the intended queue', 'priority_intended_match': 'secondary: priority equals the intended priority',
}
for k in HELD_KEYS:
    add('| ' + k + ' | ' + WHAT[k] + ' | ' + pct(dev_table, k) + ' | ' + pct(held_table, k) + ' |')
add('')
if not held_table.empty:
    add('By queue (human label), held-out. Queues with n < 5 are anecdotal, not rates:')
    _g = held_table.groupby('queue_human')[['queue_match', 'priority_match']].mean().round(2)
    _n = held_table.groupby('queue_human').size()
    for q in _g.index:
        add('- ' + q + ' (n=' + str(int(_n[q])) + '): queue_match ' + str(_g.loc[q, 'queue_match']) + ', priority_match ' + str(_g.loc[q, 'priority_match']))
    add('')
for nm in ['heldout_20', 'timing_10']:
    r = ts.get(nm, {})
    add('Time and cost (' + nm + '): ' + str(r.get('n', 'n/a')) + ' drafts traced, average ' + str(r.get('avg_latency_s', 'n/a')) + ' s, ' + str(r.get('avg_tokens_in', 'n/a')) + ' tokens in / ' + str(r.get('avg_tokens_out', 'n/a')) + ' out, total ' + str(r.get('total_cost_usd', 'n/a')) + ' USD, ' + str(r.get('total_cost_eur', 'n/a')) + ' EUR. USD is the cost LangSmith calculates from token counts (not an OpenAI invoice); EUR is an estimate: USD x ' + str(USD_TO_EUR) + ', ' + USD_TO_EUR_SOURCE + '). Source: ' + COST_SOURCE + '.')
_qa = (held_df['queue_human'] == held_df['queue_intended']).sum(); _pa = (held_df['priority_human'] == held_df['priority_intended']).sum()
add('Label context (computed from heldout_labels_v2.csv and spreekorb_emails_60.csv, not by an AI): human and intended labels agree on ' + str(_qa) + '/' + str(len(held_df)) + ' queues and ' + str(_pa) + '/' + str(len(held_df)) + ' priorities. Priority is partly a judgement call, so a low priority_match is partly label disagreement, not only AI error. The human label stays the official truth.')
add('Full table: langsmith/trace_summary.csv. Human scores: langsmith/human_review.csv.')
add('')
add('## Transparency message for Chleo')
add('- The AI never sends an email. It prepares a draft plus a suggested queue and priority. A person reads, edits and decides.')
add('- Every draft can be opened and inspected: the email, the prompt version, the policy sections it relied on, what it wrote, how long it took and what it cost.')
add('- Rules such as banned phrases, length, language and valid policy citations are checked by code, the same way every time.')
add('- We tested on invented emails only, and only 20 in the final test. This shows the method works and is visible. It does not prove quality yet.')
add('- Scores rate the AI drafts, never agents. No agent names or IDs are recorded.')
add('')
add('## Limits')
add('- Synthetic data: invented emails, not real Spreekorb customers.')
add('- Only 20 held-out emails, 1 to 2 per topic: report results for the whole set or at queue level, never per topic.')
add('- The held-out run happens once per prompt version. Any prompt change after a held-out run is a new version; the old held-out result stays as it is. Deleting a held-out experiment to rerun it is a human decision.')
add('- Code checks test form, not truth: a draft can pass all of them and still be wrong. That is why a person reviews every draft.')
add('- The banned phrase check finds exact phrases only; language is guessed from common words; time includes the network from the laptop that ran the notebook.')
add('- Cost comes from LangSmith in USD; the EUR figure is an estimate with the dated rate above.')
add('- Only synthetic emails were sent. OpenAI and LangSmith are US processors: real customer emails need a data processing agreement, a transfer basis and an EU region where offered, first. The works agreement is a draft, so this setup is prepared for compliance, not proof of it.')
add('')
add('## Evidence')
add('- LangSmith share link: ' + LANGSMITH_SHARE_URL)
if shots:
    for p in shots:
        add('- ![' + p.stem + '](screenshots/' + p.name.replace(' ', '%20') + ')')
else:
    add('- PENDING screenshot: experiment table with the evaluator columns (langsmith/screenshots/)')
    add('- PENDING screenshot: one trace showing input, output, latency, tokens and cost (langsmith/screenshots/)')
OUT['note'].write_text(chr(10).join(L) + chr(10), encoding='utf-8')
print('Wrote', OUT['note'].relative_to(REPO), len(L), 'lines')

Wrote langsmith/monitoring_note.md 73 lines


In [18]:
# 4 Verify: one PASS/FAIL per acceptance item
held_ok = not held_table.empty
check('INGEST and PROMPT match (kb_compiled.md and n8n)', INGEST_OK and PROMPT_OK)

# datasets: sizes and topics
check('dev dataset: 14 emails, 14 topics', len(dev_df) == 14 and dev_df['topic'].nunique() == 14 and client.has_dataset(dataset_name=DEV_DATASET) and len(list(client.list_examples(dataset_name=DEV_DATASET))) == 14, DEV_DATASET)
check('dev experiment has 14 drafts traced', not dev_out.empty and len(dev_out) == 14, DEV_EXP)
check('dev experiment prefix comes from the prompt version (dev-' + PROMPT_VERSION + ')', DEV_PREFIX == 'dev-' + PROMPT_VERSION and (DEV_EXP is None or DEV_EXP.startswith(DEV_PREFIX + '-')), DEV_EXP)
check('held-out labels joined: 20 rows, valid queue_human and priority_human', LABELS_OK, 'PENDING, blank or invalid: ' + str(BLANK_IDS) if not LABELS_OK else 'ok')
if held_ok:
    check('held-out dataset: 20 emails', client.has_dataset(dataset_name=HELDOUT_DATASET) and len(list(client.list_examples(dataset_name=HELDOUT_DATASET))) == 20, HELDOUT_DATASET)
    full = held_table['draft_reply'].notna() & held_table['latency_s'].notna() & held_table['tokens_in'].notna()
    check('20 held-out items traced with input, output, time, tokens, no error', len(held_table) == 20 and full.all() and held_table['error'].isna().all(), HELD_EXP)
else:
    check('held-out dataset: 20 emails traced (fix labels, then PROMPT_FROZEN = True)', False, 'PENDING')

# the 13 evaluators are present and scored (10 of the card + 3 from v5)
WANT_EVALS = ['queue_match', 'priority_match', 'priority_within_one', 'no_banned_phrase', 'cited_sections_valid', 'policy_fact_cited', 'citations_used', 'escalation_flagged', 'language_match', 'word_count', 'signoff_ok', 'no_unknown_status', 'citations_unique']
check('all 13 evaluators defined with the exact names', EVAL_KEYS == WANT_EVALS and all(k in HELD_KEYS for k in WANT_EVALS), EVAL_KEYS)
# which (email, evaluator) is blank, so a FAIL says where to look
missing_scores = lambda t, keys: [(r['email_id'], k) for _, r in t.iterrows() for k in keys if k not in t or pd.isna(r[k])] if not t.empty else ['no table']
check('dev: all 13 evaluator scores on every run (no blank, see S35 fix)', not dev_table.empty and all(k in dev_table and dev_table[k].notna().all() for k in WANT_EVALS), missing_scores(dev_table, WANT_EVALS) or 'ok')
check('held-out: all 13 evaluator scores (+ 2 secondary) on every run', held_ok and all(k in held_table and held_table[k].notna().all() for k in HELD_KEYS), missing_scores(held_table, HELD_KEYS) or 'ok')

# trace_summary.csv
tsr = pd.read_csv(OUT['trace']) if OUT['trace'].exists() else pd.DataFrame()
check('trace_summary.csv: exact columns, 2 rows (heldout_20 n=20, timing_10 n=10)', list(tsr.columns) == TS_COLS and list(tsr.get('set', [])) == ['heldout_20', 'timing_10'] and list(tsr.get('n', [])) == [20, 10], list(tsr.columns))
check('trace_summary.csv: time and cost known (R1-03 M7: n, avg_latency_s, total_cost_eur / n)', len(tsr) == 2 and bool(pd.to_numeric(tsr['total_cost_eur'], errors='coerce').notna().all()) and bool((tsr['avg_latency_s'] > 0).all()), tsr['total_cost_eur'].tolist() if len(tsr) else 'no file')

# human_review.csv
hr = pd.read_csv(OUT['human'], dtype=str) if OUT['human'].exists() else pd.DataFrame()
hr_ids = set(held_df.loc[held_df['timing_flag'], 'email_id'])
check('human_review.csv: columns email_id, language, subject, draft_reply, score_1_5, note; the 10 timing_flag drafts', list(hr.columns)[:6] == ['email_id', 'language', 'subject', 'draft_reply', 'score_1_5', 'note'] and len(hr) == 10 and set(hr['email_id']) == hr_ids)

# monitoring_note.md
note = OUT['note'].read_text(encoding='utf-8') if OUT['note'].exists() else ''
SECTIONS = ['## Why the set is synthetic', '## Prompt history v1 to v5', '## Held-out results per evaluator', '## Transparency message for Chleo', '## Limits', 'LangSmith share link:', '0 of 50']
check('monitoring_note.md has all required sections and the share link / screenshot placeholders', all(s in note for s in SECTIONS) and ('screenshot' in note.lower()), [s for s in SECTIONS if s not in note])
check('monitoring_note.md mentions v1 to v5 and queue-level reporting', all(v in note for v in ['v1', 'v2', 'v3', 'v4', 'v5', 'queue level']))
check('no key value printed or saved in langsmith/', not any(re.search(r'\bsk-[A-Za-z0-9_-]{20,}|\blsv2_[A-Za-z0-9_]{20,}', OUT[x].read_text(encoding='utf-8')) for x in OUT if OUT[x].exists()))   # real key patterns only, so 'risk-free' does not FAIL

PASS | INGEST and PROMPT match (kb_compiled.md and n8n)
PASS | dev dataset: 14 emails, 14 topics | spreekorb-dev-v4
PASS | dev experiment has 14 drafts traced | dev-v5-617203f1
PASS | dev experiment prefix comes from the prompt version (dev-v5) | dev-v5-617203f1
PASS | held-out labels joined: 20 rows, valid queue_human and priority_human | ok
PASS | held-out dataset: 20 emails | spreekorb-heldout-20
PASS | 20 held-out items traced with input, output, time, tokens, no error | heldout-v5-489260bd
PASS | all 13 evaluators defined with the exact names | ['queue_match', 'priority_match', 'priority_within_one', 'no_banned_phrase', 'cited_sections_valid', 'policy_fact_cited', 'citations_used', 'escalation_flagged', 'language_match', 'word_count', 'signoff_ok', 'no_unknown_status', 'citations_unique']
PASS | dev: all 13 evaluator scores on every run (no blank, see S35 fix) | ok
PASS | held-out: all 13 evaluator scores (+ 2 secondary) on every run | ok
PASS | trace_summary.csv: exact columns, 2

In [19]:
# 5 Unchanged: files we do not own keep the same hash
AFTER = snapshot()
changed = sorted(k for k in BEFORE if AFTER.get(k) != BEFORE[k])   # a missing file counts as changed
AFTER_IN = {k: hashlib.sha256(IN[k].read_bytes()).hexdigest() for k in INPUT_KEYS if IN[k].exists()}
check('read-only inputs unchanged (emails csv, heldout_labels_v2.csv, triage_prompt.md, kb_compiled.md)', AFTER_IN == BEFORE_IN and len(BEFORE_IN) == 4, [k for k in INPUT_KEYS if AFTER_IN.get(k) != BEFORE_IN.get(k)] or 'ok')
check('knowledgebase/ and n8n/ unchanged', not any(k.startswith(('knowledgebase', 'knowledge_base', 'n8n')) for k in changed))
check('data/processed/ unchanged (synthetic_policy_cases.csv is kept)', not any(k.startswith('data/processed') for k in changed))
check('no other watched file changed', changed == [], str(changed) if changed else 'ok, ' + str(len(AFTER)) + ' files compared')
print('(final PASS count is printed after the Undo cell)')

PASS | read-only inputs unchanged (emails csv, heldout_labels_v2.csv, triage_prompt.md, kb_compiled.md) | ok
PASS | knowledgebase/ and n8n/ unchanged
PASS | data/processed/ unchanged (synthetic_policy_cases.csv is kept)
PASS | no other watched file changed | ok, 59 files compared
(final PASS count is printed after the Undo cell)


In [ ]:
# 6 Undo (only if RUN_UNDO = True): restore backups, delete output files that did not exist before
if RUN_UNDO:
    for key, p in OUT.items():
        if existed[key]:
            shutil.copy2(BK / p.relative_to(REPO), p)
        else:
            p.unlink(missing_ok=True)
    check('undo', all(p.exists() == existed[k] for k, p in OUT.items()), 'outputs restored or removed')
    print('LangSmith datasets and experiments are not deleted here. Delete them in the LangSmith UI if needed.')
else:
    print('Undo skipped (RUN_UNDO = False)')

print()
print(sum(ok for _, ok in results), 'of', len(results), 'checks PASS')   # final count, includes undo when it ran

In [20]:
# 7 Git: print only, never run. Ayhan commits.
print('git status')
print('git add notebooks/R1-05_langsmith.ipynb langsmith/human_review.csv langsmith/trace_summary.csv langsmith/monitoring_note.md langsmith/screenshots')
print('git commit -m "R1-05 LangSmith run on the synthetic Spreekorb set"')
print('git push')

git status
git add notebooks/R1-05_langsmith.ipynb langsmith/human_review.csv langsmith/trace_summary.csv langsmith/monitoring_note.md langsmith/screenshots
git commit -m "R1-05 LangSmith run on the synthetic Spreekorb set"
git push
